# FAME Database Inspection & Auditing Tools
This notebook provides a suite of tools to quickly audit, sample, and query the compiled FAME DuckDB database.

It uses `Ibis` to push computations directly to DuckDB, meaning queries execute in milliseconds and use almost zero Python memory.

In [2]:
from pathlib import Path
from dataclasses import dataclass
import os
import sys
from dotenv import load_dotenv

# Add .env's PYTHONPATH to sys.path
load_dotenv(override=True)
PYTHONPATH = os.getenv("PYTHONPATH")
if PYTHONPATH is not None:
    if PYTHONPATH not in sys.path:
        print(f"Adding {PYTHONPATH} to sys.path")
        sys.path.append(PYTHONPATH)

# Define the structure clearly
@dataclass
class Dirs:
    data_dir: Path = None       # type: ignore
    output_dir: Path = None     # type: ignore
    input_dir: Path = None      # type: ignore
dirs = Dirs()

try:
    from utils.f_0_dirs import get_data_dirs
    dirs = get_data_dirs()
    # raise ImportError("Testing ImportError for demonstration purposes") 

# For easy access purposes, if this script is run directly in a folder with the databases
# and without the dir import modules, then just set it to current_dir
except ImportError as e:
    print(f"ImportError: {e}")
    try:
        current_dir = Path(__file__).parent
    except NameError:
        current_dir = Path.cwd()
    dirs = Dirs(output_dir=current_dir, data_dir=current_dir, input_dir=current_dir)

for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

Adding /mnt/c/Users/lazym/Documents/Code/dissertation/ to sys.path
data_dir: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30
db_path: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/fame_data.duckdb
input_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/input
output_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output
raw_data_dir: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.02
root_data_dir: /mnt/h/Other computers/My computer/fame_clean
root_dir: /mnt/c/Users/lazym/Documents/Code/dissertation
tmp_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/tmp
work_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/src


In [2]:
import pandas as pd
import ibis
import ibis.selectors as s

# 1. Setup paths
db_path = dirs.output_dir / "fame_data.duckdb"

# 2. Connect to the database
con = ibis.duckdb.connect(str(db_path))

# 3. Configure Pandas display for easier reading
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 50)

print(f"✅ Successfully connected to: {db_path}")

✅ Successfully connected to: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/fame_data.duckdb


## 1. High-Level Database Overview
Quickly check which tables exist in the database and their total row counts.

In [3]:
tables = con.list_tables()
print("📊 Database Tables Overview:\n" + "-"*30)

for table_name in tables:
    t = con.table(table_name)
    # Fast row counting pushed down to DuckDB
    row_count = t.count().execute()
    col_count = len(t.columns)
    print(f"[{table_name}]")
    print(f"   Rows: {row_count:,} | Columns: {col_count}")
print("-" * 30)

📊 Database Tables Overview:
------------------------------
[fame_derived]
   Rows: 7,948,736 | Columns: 7
[fame_fixed]
   Rows: 7,948,736 | Columns: 29
[fame_yearly]
   Rows: 40,846,539 | Columns: 37
[fame_yearly_clean]
   Rows: 40,846,539 | Columns: 37
[lars_fixed]
   Rows: 452,638 | Columns: 26
[lars_yearly]
   Rows: 2,378,089 | Columns: 35
[working_yearly_kp]
   Rows: 40,846,539 | Columns: 37
------------------------------


In [4]:
import ibis

out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
existing_tables = con.list_tables()
intersection = set(interesting_tables).intersection(existing_tables)

with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in intersection:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/duckdb_tables.md


## 2. Fast Random Sampling (Reservoir Sampling)
Extract a random subset of rows for visual inspection. We use DuckDB's native reservoir sampling (`USING SAMPLE X ROWS`) so it returns instantly, even on tables with millions of rows, without doing a full table scan.

In [5]:
target_table = "fame_yearly"
sample_frac = 0.001  # Sample fraction for random sampling

# Native Ibis sampling using positional row count and seed parameter
table = con.table(target_table)
row_count = table.count().execute()
df_sample = table.sample(sample_frac, seed=12345).execute()

print(f"🎲 Random sample of {row_count:,} rows from '{target_table}':")
display(df_sample)

🎲 Random sample of 40,846,539 rows from 'fame_yearly':


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,tangibles_land_leasehold,tangibles_fixt_fit,tangibles_plant_and_vehicles,tangibles_plant,tangibles_vehicles,fixed_other,intangibles,investments_other,fixed_total,liabilities,total_assets,liabilites_lt,cos,admin_expenses,interest_paid,profit_loss_pretax2,tax,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,10005873,2020,False,NaN,2.553000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.712000,7.265000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,08153130,2020,False,NaN,1146.638000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-73.749000,1220.387000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,04631669,2009,False,3.450000,39.320000,-2.15400,NaN,36.140000,36.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.140000,-0.375000,39.695000,NaN,NaN,-5.619000,NaN,-2.15400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.16900
3,IE255973,2006,False,219.555775,17.803939,14.70154,8.0,2.674043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.674043,-26.639209,44.443148,NaN,NaN,-204.854236,NaN,14.70154,NaN,NaN,1.68393,NaN,131.525521,113.606951,9.691312,8.227258,NaN,NaN,16.38547
4,IE527336,2017,False,NaN,-9.082505,NaN,NaN,1.566677,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.566677,-25.926607,16.844102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.431689,25.431689,NaN,NaN,NaN,4.745638,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40821,03259238,2014,False,0.429000,2.709000,0.42900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.709000,-3.000,NaN,NaN,NaN,0.42900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.42900
40822,IE415381,2020,False,NaN,712.413306,NaN,NaN,78.511327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78.511327,NaN,NaN,78.511327,-140.980592,853.393898,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40823,06163560,2019,False,NaN,333.173000,NaN,NaN,682.295000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,682.295000,NaN,NaN,682.295000,-218.907000,684.794000,-132.714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40824,08385687,2021,False,NaN,77.072000,NaN,NaN,242.314000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,242.314000,NaN,NaN,242.314000,-179.071000,256.833000,-0.690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Specific Lookup

### Firm-specific (Search by Name or ID)
Find all fixed and derived attributes for a specific company using partial string matching (case-insensitive).

In [6]:
search_term = "TESCO"  # Can be a partial name or a Registered Number
table_fixed = con.table("fame_fixed")

# Filter where company_name contains the search term OR exactly matches registered_number
search_query = table_fixed.filter(
    table_fixed["company_name"].upper().contains(search_term.upper()) |
    (table_fixed["registered_number"] == search_term)
)

# Pull the top 5 matches
df_search_results = search_query.head(10).execute()

print(f"🔍 Top search results for '{search_term}':")
display(df_search_results[["registered_number", "company_name", "ro_address", "primary_trading_address"]])

🔍 Top search results for 'TESCO':


,registered_number,company_name,ro_address,primary_trading_address
0,08255488,TESCO DORNEY (1LP) LIMITED,"Tesco House Shire Park, Kestrel Way, Welwyn Ga...","Tesco House Shire Park, Kestrel Way, Welwyn Ga..."
1,08255493,TESCO DORNEY (GP) LIMITED,"Tesco House Shire Park, Shires Park, Kestrel W...","Tesco House Shire Park, Shires Park, Kestrel W..."
2,08255503,TESCO DORNEY (NOMINEE HOLDCO) LIMITED,"Tesco House Shire Park, Shires Park, Kestrel W...","Tesco House Shire Park, Shires Park, Kestrel W..."
3,08255640,TESCO DORNEY (NOMINEE 1) LIMITED,"Tesco House Shire Park, Kestrel Way, Welwyn Ga...","Tesco House Shire Park, Kestrel Way, Welwyn Ga..."
4,08255645,TESCO DORNEY (NOMINEE 2) LIMITED,"Tesco House Shire Park, Kestrel Way, Welwyn Ga...","Tesco House Shire Park, Kestrel Way, Welwyn Ga..."
5,08339162,MITESCO LIMITED,"29 Oakwood Park Road, London, N14 6QB","29 Oakwood Park Road, London, London, N14 6QB"
6,SC208935,TESCO CORPORATION (UK) LIMITED,"13 Queen's Road, Aberdeen, Aberdeenshire, AB15...",NaN
7,08859202,TESCO FFC LIMITED,"1 More London Place, London, SE1 2AF",NaN
8,05349394,TESCO PROPERTY (NOMINEES) (NO.3) LIMITED,"Tesco House Shire Park, Kestrel Way, Welwyn Ga...","Tesco House Shire Park, Kestrel Way, Welwyn Ga..."
9,05349395,TESCO PROPERTY (NOMINEES) (NO.4) LIMITED,"Tesco House Shire Park, Kestrel Way, Welwyn Ga...","Tesco House Shire Park, Kestrel Way, Welwyn Ga..."


### Industry-specific (by 2-digit SIC or 6-digit
- 6-digit using primary_uk_sic_2007_code in fame_fixed

In [7]:
# ### Industry-specific (by 2-digit SIC or 6-digit
# - 6-digit using primary_uk_sic_2007_code in fame_fixed
import json
import ibis
import pandas as pd

# Read build/input/SIC_priorities.json
search_inds = []

get_from_json = False
if get_from_json:
    with open(dirs.input_dir / "SIC_priorities.json", "r") as f:
        sic_priorities = json.load(f)
        search_inds = [entry.get("Division") for entry in sic_priorities if entry.get("Errored") == True]
else:
    search_inds = ["70", "74"]

# Ensure strings are properly formatted (e.g. padded with zeros) just in case
search_inds_str = [str(x).zfill(2) for x in search_inds if x is not None]
print(f"Searching for following SIC divisions individually: {search_inds_str}")

# Connect to the tables in the database
t_fixed = con.table("fame_fixed")
t_derived = con.table("fame_derived")
t_yearly = con.table("fame_yearly")

# List to accumulate our row records
results = []

for ind in search_inds_str:
    # 1. Filter the fame_derived table for the specific SIC division
    regex_pattern = rf'\b{ind}\b'
    filtered_derived = t_derived.filter(
        t_derived.industry_codes.re_search(regex_pattern)
    )

    # 2. Execute the count for fame_derived
    derived_count = filtered_derived.count().execute()

    # 3. Filter fame_fixed using a semi-join
    fixed_count = t_fixed.semi_join(filtered_derived, "registered_number").count().execute()

    # 4. Filter fame_yearly using a semi-join
    yearly_count = t_yearly.semi_join(filtered_derived, "registered_number").count().execute()

    # Append the counts for this specific division to our results list
    results.append({
        "SIC_Division": ind,
        "fame_derived": derived_count,
        "fame_fixed": fixed_count,
        "fame_yearly": yearly_count
    })

# Output the results as a formatted table
output_table = pd.DataFrame(results)

print("📊 Matching Rows by Table for Each SIC Division:")
display(output_table) # use print(output_table) if you aren't in a Jupyter environment

Searching for following SIC divisions individually: ['70', '74']
📊 Matching Rows by Table for Each SIC Division:


,SIC_Division,fame_derived,fame_fixed,fame_yearly
0,70,596707,760150,3347646
1,74,235756,314169,298244


## 4. Time-Series Construction (Joining Fixed & Yearly)
Combine the static company metadata with its longitudinal financial performance. This demonstrates the relational integrity of the `registered_number` primary key.

In [8]:
# Pick a specific company ID to track over time
selected_ids = df_sample["registered_number"].head().tolist()

t_fixed = con.table("fame_fixed")
t_yearly = con.table("fame_yearly")

# 1. Filter both tables to the target ID
firm_fixed = t_fixed.filter(t_fixed["registered_number"].isin(selected_ids))
firm_yearly = t_yearly.filter(t_yearly["registered_number"].isin(selected_ids))

# 2. Left join yearly financials onto the fixed metadata
firm_history = firm_yearly.left_join(firm_fixed, "registered_number")

# 3. Select a curated subset of columns to display
comprehensive_view = firm_history.select(
    "registered_number",
    "company_name",
    "year",
    # Safely select financial columns if they exist in the DB
    s.contains("turnover"),
    s.contains("profit_loss_pretax"),
    s.contains("employees")
).order_by("year") # Order chronologically

display(comprehensive_view.execute())

,registered_number,company_name,year,turnover,profit_loss_pretax,profit_loss_pretax2,employees,remuneration_employees
0,04631669,SPENCER PARK (BLOCK A) FREEHOLD LIMITED,2006,0.200000,-0.531000,-0.531000,NaN,NaN
1,IE255973,NaN,2006,219.555775,14.701540,14.701540,8.0,131.525521
2,04631669,SPENCER PARK (BLOCK A) FREEHOLD LIMITED,2007,0.300000,0.018000,0.018000,NaN,NaN
3,IE255973,NaN,2007,393.716045,30.214970,30.214970,18.0,274.229507
4,IE255973,NaN,2008,488.910428,-39.939987,-39.939987,21.0,398.363115
5,04631669,SPENCER PARK (BLOCK A) FREEHOLD LIMITED,2008,1.840000,1.522000,1.522000,NaN,NaN
6,IE255973,NaN,2009,512.671788,12.183068,12.183068,21.0,388.546999
7,04631669,SPENCER PARK (BLOCK A) FREEHOLD LIMITED,2009,3.450000,-2.154000,-2.154000,NaN,NaN
8,IE255973,NaN,2010,511.214544,-3.464098,-3.464098,23.0,392.165764
9,04631669,SPENCER PARK (BLOCK A) FREEHOLD LIMITED,2010,0.200000,-0.655000,-0.655000,NaN,NaN


## 5. Summary Statistics & Aggregations
Generate high-level analytical cuts (e.g., counting the number of records per year, or assessing data coverage).

In [ ]:
t_yearly = con.table("fame_yearly")

# Aggregate the number of financial records available per year
# Print numbers of available records for every financial column
yearly_distribution = (
    t_yearly
    .group_by("year")
    .aggregate(
        turnover=t_yearly["turnover"].count(),
        pnl=t_yearly["profit_loss_pretax"].count(),
        employees=t_yearly["employees"].count(),
        tangibles=t_yearly["tangibles"].count(),
        t_lab=t_yearly["tangibles_land_and_buildings"].count(),
        t_land_free=t_yearly["tangibles_land_freehold"].count(),
        t_land_lease=t_yearly["tangibles_land_leasehold"].count(),
        fixed_other=t_yearly["fixed_other"].count(),
        intangibles=t_yearly["intangibles"].count(),
        fixed_total=t_yearly["fixed_total"].count(),
        liabilities=t_yearly["liabilities"].count(),
        t_assets=t_yearly["total_assets"].count(),
        liabilites_lt=t_yearly["liabilites_lt"].count(),
        cos=t_yearly["cos"].count(),
        dividends=t_yearly["dividends"].count(),
        r_and_d=t_yearly["r_and_d"].count(),
        renum=t_yearly["remuneration_employees"].count(),
        wages=t_yearly["wages"].count(),
        ss_cost=t_yearly["social_security_costs"].count(),
        pension_cost=t_yearly["pensions_costs"].count(),
        ebitda=t_yearly["ebitda"].count(),
        all=t_yearly.count()
    )
    .order_by(ibis.desc("year"))
)

print("📈 Data coverage by year:")
# Format table with commas for readability and display
count_table: pd.DataFrame = yearly_distribution.execute()
count_table_formatted = count_table.copy().reset_index(drop=True)
for col in count_table_formatted.columns:
    if col != "year":
        count_table_formatted[col] = count_table_formatted[col].apply(lambda x: f"{x:,}")
display(count_table_formatted)

📈 Data coverage by year:


,year,turnover,pnl,employees,tangibles,t_lab,t_land_free,t_land_lease,fixed_other,intangibles,fixed_total,liabilities,t_assets,liabilites_lt,cos,dividends,r_and_d,renum,wages,ss_cost,pension_cost,ebitda,all
0,2025,727,824,"8,198","7,659",449,320,146,"5,524",484,"10,532","11,914","17,305","5,861",1,0,0,0,0,0,0,2,"22,483"
1,2024,"82,705","92,932","1,079,271","837,780","91,922","75,073","20,739","574,913","50,765","1,025,698","1,247,585","1,549,747","655,363","33,048","11,699","1,374","51,642","40,147","32,419","33,231","94,577","1,722,640"
2,2023,"201,195","235,246","1,978,444","1,648,448","210,559","165,775","54,730","1,105,499","124,602","2,036,562","2,475,027","3,046,806","1,334,746","93,664","31,953","5,442","126,560","107,202","88,585","87,289","231,645","3,154,094"
3,2022,"201,882","236,256","1,937,296","1,613,612","207,486","162,065","55,684","1,084,650","124,539","1,986,537","2,422,241","2,979,283","1,315,217","93,386","30,697","5,445","125,747","106,299","87,897","85,801","232,609","3,082,677"
4,2021,"200,675","234,549","1,882,335","1,567,015","200,119","155,330","54,428","1,050,696","124,239","1,915,718","2,365,058","2,894,373","1,277,843","91,144","29,558","5,144","126,261","106,317","86,805","84,012","231,245","2,991,930"
5,2020,"201,791","233,830","1,803,749","1,503,738","196,597","153,226","53,239","1,011,213","123,320","1,817,547","2,292,799","2,767,373","1,076,973","90,589","30,057","5,020","125,246","110,931","86,489","82,165","230,577","2,859,525"
6,2019,"203,521","232,814","1,336,996","1,434,052","201,105","161,831","50,523","985,004","123,372","1,725,490","2,183,798","2,627,691","888,754","90,514","33,714","5,041","123,481","122,055","85,167","78,447","229,693","2,713,618"
7,2018,"205,771","234,458","1,117,333","1,379,959","186,267","150,817","45,771","899,585","125,247","1,649,540","2,079,646","2,500,845","836,985","92,742","36,685","4,609","126,608","125,645","85,443","76,820","239,570","2,582,968"
8,2017,"216,217","246,792","946,294","1,332,132","180,974","147,160","43,590","854,605","130,445","1,565,378","2,002,573","2,395,443","784,325","97,247","40,551","4,470","140,463","139,446","92,034","76,584","257,266","2,472,123"
9,2016,"202,371","225,484","527,435","1,275,645","89,651","67,292","29,888","1,058,811","162,320","1,501,073","1,898,217","2,267,469","675,977","96,635","45,206","3,501","153,082","151,487","107,741","78,635","230,664","2,334,767"


## 6. Exporting Queries to Excel
Any queried subset of data can be instantly dumped into an Excel file for offline review.

In [3]:
import pandas as pd
import ibis

approach = 'random' # first

# Dump a random sample of 500 rows from each table to a single .xlsx file in /tmp
row_count = 2000
out_file_raw = dirs.output_dir / "df_raw_sample.xlsx"
desired_tables = ["fame_derived", "fame_fixed", "fame_yearly_filtered", "lars_fixed", "lars_yearly"]

# but we want to have 5 separate tabs in the same excel file, one for each table, with the table name as the tab name
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    # df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    for table in desired_tables:
        if approach == 'first':
            con.table(table).head(row_count).execute().to_excel(writer, sheet_name=table, index=False)
        elif approach == 'random':
            nb_rows = con.table(table).count().execute()
            fraction = row_count / nb_rows if nb_rows > row_count else 1.0
            df_table_rd = con.table(table).sample(fraction, seed=12345).execute()
            df_table_rd.to_excel(writer, sheet_name=table, index=False)
        
print(f"✅ Successfully dumped a random {row_count} rows of df_raw to: {out_file_raw}")

✅ Successfully dumped a random 2000 rows of df_raw to: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/df_raw_sample.xlsx


# 7. Industry counts in each of the tables
### [read/(write)] Count cells belong to certain industries
- Using the fame_derived system
- Remove rows belonging to a certain industry, as a 'fresh start' mechanism

In [2]:
# connect to fame_derived and group by industry_codes
# count the number of rows belonging to each group (unique industry_codes)
# Sort the results in ascending order of industry_codes
# Output the result (as a table, with counts as the column)
import ibis
from utils.f_0_dirs import get_data_dirs

con = ibis.duckdb.connect(str(dirs.db_path))
dirs = get_data_dirs()
fame_derived = con.table("fame_derived")
industry_counts = (
    fame_derived
    .group_by("industry_codes")
    .aggregate(count=fame_derived.count())
    .order_by("industry_codes")
)
# Execute and display the results
df_industry_counts = industry_counts.execute()
print("📊 Industry Codes and Their Counts:")
for index, row in df_industry_counts.iterrows():
    print(f"Industry Code: {row['industry_codes']}, Count: {row['count']:,}")

📊 Industry Codes and Their Counts:
Industry Code: 01, Count: 42,142
Industry Code: 02, Count: 7,744
Industry Code: 03, Count: 6,573
Industry Code: 05, Count: 444
Industry Code: 06, Count: 4,445
Industry Code: 07, Count: 1,036
Industry Code: 08, Count: 3,748
Industry Code: 09, Count: 11,152
Industry Code: 10, Count: 33,141
Industry Code: 11, Count: 9,878
Industry Code: 12, Count: 294
Industry Code: 13, Count: 13,800
Industry Code: 14, Count: 19,895
Industry Code: 15, Count: 3,526
Industry Code: 16, Count: 16,828
Industry Code: 17, Count: 6,168
Industry Code: 18, Count: 31,021
Industry Code: 19, Count: 781
Industry Code: 20, Count: 11,451
Industry Code: 21, Count: 3,259
Industry Code: 22, Count: 12,727
Industry Code: 23, Count: 9,020
Industry Code: 24, Count: 6,164
Industry Code: 25, Count: 49,205
Industry Code: 26, Count: 19,531
Industry Code: 27, Count: 10,650
Industry Code: 28, Count: 20,683
Industry Code: 29, Count: 9,243
Industry Code: 30, Count: 9,196
Industry Code: 31, Count: 18,1

In [ ]:
run_this_cell = True
run_overwrite = False

if not run_this_cell:
    raise ValueError("Execution skipped as per the run_this_cell flag.")

import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

error_industries = ['47', '74', '43']

dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(dirs.db_path))

table_fame_derived = con.table("fame_derived")
table_fame_fixed = con.table("fame_fixed")
table_fame_yearly = con.table("fame_yearly")

# Helper function to prevent repeating the same counting block 3 times
def get_table_counts(derived, fixed, yearly, count_col_name="count"):
    return pd.DataFrame({
        "table": ["fame_derived", "fame_fixed", "fame_yearly"],
        count_col_name: [derived.count().execute(), fixed.count().execute(), yearly.count().execute()]
    })

count_before_df = get_table_counts(table_fame_derived, table_fame_fixed, table_fame_yearly, "count_before")
print(f"✅ Counts before filtering:\n{count_before_df.to_markdown(index=False)}\n")

# ==========================================
# 1. FILTERED (Firms IN error_industries)
# ==========================================
table_fame_derived_filtered = table_fame_derived.filter(table_fame_derived.industry_codes.isin(error_industries))

# SEMI-JOIN: Keeps rows in the left table that have a match in the right table.
# This eliminates the need for an inner_join + manual .select()
table_fame_fixed_filtered = table_fame_fixed.semi_join(table_fame_derived_filtered, "registered_number")
table_fame_yearly_filtered = table_fame_yearly.semi_join(table_fame_derived_filtered, "registered_number")

count_errors_df = get_table_counts(table_fame_derived_filtered, table_fame_fixed_filtered, table_fame_yearly_filtered, "count_after")
print(f"✅ Counts after filtering:\n{count_errors_df.to_markdown(index=False)}\n")

# ==========================================
# 2. EXCLUDED (Firms NOT IN error_industries)
# ==========================================
table_fame_derived_excluded = table_fame_derived.filter(~table_fame_derived.industry_codes.isin(error_industries))

# ANTI-JOIN: Keeps rows in the left table that DO NOT have a match in the right table.
# This replaces the cumbersome left_join + isnull() filter + select()
table_fame_fixed_excluded = table_fame_fixed.anti_join(table_fame_derived_filtered, "registered_number")
table_fame_yearly_excluded = table_fame_yearly.anti_join(table_fame_derived_filtered, "registered_number")

count_excluded_df = get_table_counts(table_fame_derived_excluded, table_fame_fixed_excluded, table_fame_yearly_excluded, "count_excluded")
print(f"✅ Counts after exclusion:\n{count_excluded_df.to_markdown(index=False)}\n")

# ==========================================
# 3. VERIFICATION
# ==========================================
# Use Pandas vectorized math to check all three tables simultaneously
expected_counts = count_before_df["count_before"] - count_errors_df["count_after"]
matches = count_excluded_df["count_excluded"] == expected_counts

if not matches.all():
    failed_tables = count_excluded_df.loc[~matches, "table"].tolist()
    raise ValueError(f"❌ Excluded counts do not match expected counts for: {', '.join(failed_tables)}")

for _, row in count_excluded_df.iterrows():
    t_name = row["table"]
    print(f"✅ Excluded counts match expected counts for table {t_name}: {row['count_excluded']:,} = "
          f"{count_before_df.loc[count_before_df['table'] == t_name, 'count_before'].values[0]:,} - "
          f"{count_errors_df.loc[count_errors_df['table'] == t_name, 'count_after'].values[0]:,}")

# ==========================================
# 4. SAFE DATABASE OVERWRITE (ATOMIC SWAP)
# ==========================================
if run_overwrite:
    print("💾 Writing clean data to temporary tables and performing atomic swaps...")

    # 1. Update fame_derived
    # Check if the exluded count is zero
    if count_errors_df.loc[count_errors_df['table'] == 'fame_derived', 'count_after'].values[0] > 0: # type: ignore
        con.create_table("fame_derived_clean", table_fame_derived_excluded, overwrite=True)
        con.drop_table("fame_derived")
        con.create_table("fame_derived", con.table("fame_derived_clean"), overwrite=True)
        print("✅ Successfully updated fame_derived table.")
    else:
        print("⚠️ Excluded count for fame_derived is not zero. Skipping update to fame_derived table.")

    # 2. Update fame_fixed
    if count_errors_df.loc[count_errors_df['table'] == 'fame_fixed', 'count_after'].values[0] > 0: # type: ignore
        con.create_table("fame_fixed_clean", table_fame_fixed_excluded, overwrite=True)
        con.drop_table("fame_fixed")
        con.create_table("fame_fixed", con.table("fame_fixed_clean"), overwrite=True)
        print("✅ Successfully updated fame_fixed table.")
    else:
        print("⚠️ Excluded count for fame_fixed is not zero. Skipping update to fame_fixed table.")

    # 3. Update fame_yearly
    if count_errors_df.loc[count_errors_df['table'] == 'fame_yearly', 'count_after'].values[0] > 0: # type: ignore
        con.create_table("fame_yearly_clean", table_fame_yearly_excluded, overwrite=True)
        con.drop_table("fame_yearly")
        con.create_table("fame_yearly", con.table("fame_yearly_clean"), overwrite=True)
        print("✅ Successfully updated fame_yearly table.")
    else:
        print("⚠️ Excluded count for fame_yearly is not zero. Skipping update to fame_yearly table.")

TableNotFound: fame_derived